<a href="https://colab.research.google.com/github/Pranayshukla0610/Natural-Language-Processing-NLP-/blob/main/NLP_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
#Getting dataset
df = pd.read_csv('/content/mushrooms.csv')

In [ ]:
df.shape

(8124, 23)

In [ ]:
df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [ ]:
le = LabelEncoder()

In [ ]:
df_encoded = df.apply(le.fit_transform,axis=0)

In [ ]:
df_encoded.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,1,5,2,4,1,6,1,0,1,4,...,2,7,7,0,2,1,4,2,3,5
1,0,5,2,9,1,0,1,0,0,4,...,2,7,7,0,2,1,4,3,2,1
2,0,0,2,8,1,3,1,0,0,5,...,2,7,7,0,2,1,4,3,2,3
3,1,5,3,8,1,6,1,0,1,5,...,2,7,7,0,2,1,4,2,3,5
4,0,5,2,3,0,5,1,1,0,4,...,2,7,7,0,2,1,0,3,0,1


In [ ]:
df = df_encoded.values

In [ ]:
X = df[:, 1:]

In [ ]:
y = df[:,0]

In [ ]:
X

array([[5, 2, 4, ..., 2, 3, 5],
       [5, 2, 9, ..., 3, 2, 1],
       [0, 2, 8, ..., 3, 2, 3],
       ...,
       [2, 2, 4, ..., 0, 1, 2],
       [3, 3, 4, ..., 7, 4, 2],
       [5, 2, 4, ..., 4, 1, 2]])

In [ ]:
y

array([1, 0, 0, ..., 0, 1, 0])

In [ ]:
X_train, y_train, X_test, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

#NAIVE BAYES CLASSIFIER
##Posterior Probability
##Likelihood Probability
##Prior Probability

In [ ]:
def fix_shape(X):
    X = np.array(X)
    if len(X.shape) == 1:        # if shape = (n,)
        X = X.reshape(-1, 1)     # reshape to (n,1)
    return X

In [ ]:

X_train = fix_shape(X_train)
X_test  = fix_shape(X_test)

# Make sure both have same number of features
if X_train.shape[1] != X_test.shape[1]:
    X_test = X_test.reshape(-1, X_train.shape[1])

ValueError: cannot reshape array of size 6499 into shape (22)

In [ ]:
# Prior Probability: P(label)
def prior_prob(y, label):
    m = y.shape[0]
    s = np.sum(y == label)
    return s / m

In [ ]:
# Conditional Probability: P(feature_val | label) with Laplace smoothing
def cond_prob(X_train, y_train, feature_col, feature_val, label):
    X_filtered = X_train[y_train == label]
    num = np.sum(X_filtered[:, feature_col] == feature_val)

    # Laplace smoothing
    denom = X_filtered.shape[0]
    return (num + 1) / (denom + 2)   # "+2" since binary/multi-category features

In [ ]:
# Predict one sample
def predict(X_train, y_train, x):
    classes = np.unique(y_train)
    n_features = X_train.shape[1]
    posterior_prob = []

    for label in classes:
        likelihood = 1.0

        for fea in range(n_features):
            cond = cond_prob(X_train, y_train, fea, x[fea], label)
            likelihood *= cond

        prior = prior_prob(y_train, label)
        posterior = likelihood * prior
        posterior_prob.append(posterior)

    return np.argmax(posterior_prob)




In [ ]:
# Accuracy
def accuracy(X_train, y_train, X_test, y_test):
    pred = []
    for i in range(len(X_test)):
        pred.append(predict(X_train, y_train, X_test[i]))

    pred = np.array(pred)
    return np.mean(pred == y_test)


In [ ]:
acc = accuracy(X_train, y_train, X_test, y_test)
acc